# PoseCoach P02 — Fit3D Steps 11b → 12 (Kaggle Edition)

> Reads your Fit3D data **directly from Google Drive** via PyDrive2.
> No `google.colab` imports — works 100% in Kaggle.

---
## ⚙️ One-Time Setup (do this BEFORE running any cell)

You need two Kaggle Secrets containing Google OAuth2 credentials.
This takes about **5 minutes** and only needs to be done once.

### Step A — Create Google OAuth2 credentials
1. Go to **https://console.cloud.google.com/** (sign in with your Google account)
2. Click the project dropdown (top bar) → **New Project** → name it anything → **Create**
3. In the left menu: **APIs & Services → Library** → search **"Google Drive API"** → click it → **Enable**
4. In the left menu: **APIs & Services → Credentials** → **+ Create Credentials → OAuth client ID**
   - If it asks to configure the consent screen first: choose **External**, fill in just the App name (anything) and your email, then **Save and Continue** through all steps
5. Application type: **Desktop app** → **Create**
6. A popup shows your **Client ID** and **Client Secret** — copy both

### Step B — Add secrets to Kaggle
In your Kaggle notebook:
1. Click the **three-dot menu (⋮)** at the top right → **Add-ons → Secrets**
2. Add two secrets (click **+ Add a new secret** for each):
   - Name: `GDRIVE_CLIENT_ID`     → Value: paste your Client ID from Step A
   - Name: `GDRIVE_CLIENT_SECRET` → Value: paste your Client Secret from Step A
3. **Toggle both secrets ON** for this notebook

### Step C — Auth flow (every new Kaggle session)
When you run the Auth cell below, it prints a URL.
Open that URL in your browser, sign in with Google, click Allow, and **copy the verification code** back into the cell's input box.

In [ ]:
# ── Install PyDrive2 (not pre-installed in Kaggle) ────────────────────────────
!pip install pydrive2 -q
print('✅ pydrive2 installed')

In [ ]:
# ── Google Drive Authentication ───────────────────────────────────────────────
# Reads your credentials from Kaggle Secrets and opens a browser auth flow.

import json, os
from kaggle_secrets import UserSecretsClient

secrets       = UserSecretsClient()
client_id     = secrets.get_secret('GDRIVE_CLIENT_ID')
client_secret = secrets.get_secret('GDRIVE_CLIENT_SECRET')

# Write temporary credentials file
os.makedirs('/tmp', exist_ok=True)
with open('/tmp/client_secrets.json', 'w') as f:
    json.dump({
        'installed': {
            'client_id':     client_id,
            'client_secret': client_secret,
            'auth_uri':      'https://accounts.google.com/o/oauth2/auth',
            'token_uri':     'https://oauth2.googleapis.com/token',
            'redirect_uris': ['urn:ietf:wg:oauth:2.0:oob']
        }
    }, f)

from pydrive2.auth import GoogleAuth
from pydrive2.drive import GoogleDrive

gauth = GoogleAuth()
gauth.DEFAULT_SETTINGS['client_config_file'] = '/tmp/client_secrets.json'
gauth.DEFAULT_SETTINGS['oauth_scope']        = ['https://www.googleapis.com/auth/drive.readonly']
gauth.DEFAULT_SETTINGS['save_credentials']   = False

# Prints a URL → open it → sign in → copy the code → paste it in the input box below
gauth.CommandLineAuth()
drive = GoogleDrive(gauth)
print('\n✅ Connected to Google Drive!')

In [ ]:
# ── Download joints3d_25 and rep_ann.json for all 8 train subjects ────────────
# Downloads ONLY the joint JSON files and rep annotations (~750 MB total).
# Videos, smplx, and camera files are skipped entirely.
#
# These IDs were pre-looked-up from your Drive — no manual folder sharing needed.

import os, time

FIT3D_DIR = '/kaggle/working/fit3d'

# All IDs sourced directly from your Google Drive
SUBJECTS = {
    's03': {
        'joints3d_25_folder': '1IlKCyn5Rregnn0dIIoUrhKnY8VbnOmX9',
        'rep_ann_file':       '1pMS57-vVO3tUTHgu1teqvMpl-poi6hqk',
    },
    's04': {
        'joints3d_25_folder': '1ukwh4A-pRSar77aSvJRCF2Glmyi-FY6k',
        'rep_ann_file':       '1H7nc975GrX5W5vQdlzfMfmjky8HrfXCj',
    },
    's05': {
        'joints3d_25_folder': '1Hfa3CjNV0QvEXlaRHFkjJ-YzZiA8DE9k',
        'rep_ann_file':       '1npg4BHqBZxQc7gFR-QUJT2lUEl-493AY',
    },
    's07': {
        'joints3d_25_folder': '1F1V6Tqfa8Voi-zE1p4BrsvWJtCItyOlw',
        'rep_ann_file':       '1_lkWlLLQB0aHtL6bH0TQqpcu1W4U3zg8',
    },
    's08': {
        'joints3d_25_folder': '1f1bjE0fOhzRflyPzSWFz2Qm4C426_XzH',
        'rep_ann_file':       '1eZYlCI2IPrToLrS-U2u1I3Emg4dC5Wou',
    },
    's09': {
        'joints3d_25_folder': '14r6iTeOvkw2GckkGembCUWStxA9k41lX',
        'rep_ann_file':       '1t_LzR2v6S10B1dNT2PIOXE02FJ7XhWoh',
    },
    's10': {
        'joints3d_25_folder': '1LAf_4Wl6oqu23Vg8cCSIlpfNOp0wRk_P',
        'rep_ann_file':       '1TvTRmk2c4JLov7pgHrEywBSDokOmWlj1',
    },
    's11': {
        'joints3d_25_folder': '1jEj_P2TU-EPjImU5Du2fZRMAPMvQXRLq',
        'rep_ann_file':       '1yDLj9N5RYTXAXEUC-OojsxHAMoTjWAQU',
    },
}

FIT3D_INFO_FILE_ID = '18INFmXFy-gnqVtV2u7D1XQZESTj9-hdN'


def download_drive_folder(drive, folder_id, local_dir, ext='.json', skip_existing=True):
    """Download all files with given extension from a Drive folder into local_dir."""
    os.makedirs(local_dir, exist_ok=True)
    items = drive.ListFile({'q': f"'{folder_id}' in parents and trashed=false"}).GetList()
    files = [f for f in items if f['title'].endswith(ext)]
    for f in files:
        dest = os.path.join(local_dir, f['title'])
        if skip_existing and os.path.exists(dest):
            continue
        f.GetContentFile(dest)
    return len(files)


def download_drive_file(drive, file_id, local_path, skip_existing=True):
    """Download a single file from Drive."""
    if skip_existing and os.path.exists(local_path):
        return
    os.makedirs(os.path.dirname(local_path), exist_ok=True)
    f = drive.CreateFile({'id': file_id})
    f.GetContentFile(local_path)


# ── Download fit3d_info.json ──────────────────────────────────────────────────
info_path = f'{FIT3D_DIR}/fit3d_info.json'
download_drive_file(drive, FIT3D_INFO_FILE_ID, info_path)
print('✅ fit3d_info.json')

# ── Download per-subject data ─────────────────────────────────────────────────
t0 = time.time()
for subj, ids in SUBJECTS.items():
    j3d_dir      = f'{FIT3D_DIR}/{subj}/joints3d_25'
    rep_ann_path = f'{FIT3D_DIR}/{subj}/rep_ann.json'

    n = download_drive_folder(drive, ids['joints3d_25_folder'], j3d_dir)
    download_drive_file(drive, ids['rep_ann_file'], rep_ann_path)
    print(f'  {subj}: {n} exercise JSONs + rep_ann.json  ✅')

elapsed = time.time() - t0
print(f'\n✅ All data downloaded in {elapsed/60:.1f} min')
print(f'   Location: {FIT3D_DIR}/')

---
## ✅ Step 11b — Explore Fit3D Dataset Structure

In [ ]:
import json
from pathlib import Path
from collections import defaultdict

fit3d_root = Path(FIT3D_DIR)

with open(f'{FIT3D_DIR}/fit3d_info.json') as f:
    fit3d_info = json.load(f)
print('fit3d_info.json keys:', list(fit3d_info.keys()))

exercise_set = set()
print('\n=== Dataset Structure ===')
for subj in sorted(SUBJECTS.keys()):
    j3d_dir = fit3d_root / subj / 'joints3d_25'
    rep_ann = fit3d_root / subj / 'rep_ann.json'
    exercises = sorted(j3d_dir.glob('*.json'))
    exercise_set.update(f.stem for f in exercises)
    rep_count = len(json.load(open(rep_ann))) if rep_ann.exists() else 0
    print(f'  {subj}:  {len(exercises)} exercises  |  {rep_count} rep-annotated sequences')

print(f'\nTotal unique exercises: {len(exercise_set)}')
for ex in sorted(exercise_set):
    print(f'  {ex}')

In [ ]:
# ── Inspect one JSON file to confirm the joint data format ────────────────────
import numpy as np

sample_file = sorted((Path(FIT3D_DIR) / 's03' / 'joints3d_25').glob('*.json'))[0]
with open(sample_file) as f:
    sample_data = json.load(f)

print(f'Sample: {sample_file.name}')
print(f'Top-level keys: {list(sample_data.keys())}')

# Auto-detect the 3D joints array
JOINTS_KEY = None
for key, val in sample_data.items():
    if isinstance(val, list):
        arr = np.array(val)
        if arr.ndim == 3 and arr.shape[-1] == 3:
            JOINTS_KEY = key
            print(f'\n✅ 3D joints found under key "{key}"')
            print(f'   Shape: {arr.shape}  →  [N_frames={arr.shape[0]}, N_joints={arr.shape[1]}, xyz=3]')
            print(f'   Range: [{arr.min():.3f}, {arr.max():.3f}] meters')
            break

if JOINTS_KEY is None:
    print('⚠️  Could not auto-detect joints key. Keys and shapes:')
    for k, v in sample_data.items():
        print(f'  {k}: {np.array(v).shape if isinstance(v, list) else type(v)}')
    JOINTS_KEY = 'joints3d_25'  # fallback

---
## ✅ Step 11c — Compute Golden Angle Templates

Fit3D 25-joint skeleton (SMPL-X body subset):
```
 0:pelvis  1:l_hip   2:r_hip   3:spine1  4:l_knee  5:r_knee
 6:spine2  7:l_ankle 8:r_ankle 9:spine3 10:l_foot 11:r_foot
12:neck   13:l_collar 14:r_collar 15:head 16:l_shoulder 17:r_shoulder
18:l_elbow 19:r_elbow 20:l_wrist 21:r_wrist 22:l_hand 23:r_hand 24:jaw
```

In [ ]:
import numpy as np

JOINT_NAMES = [
    'pelvis','l_hip','r_hip','spine1','l_knee','r_knee','spine2',
    'l_ankle','r_ankle','spine3','l_foot','r_foot','neck',
    'l_collar','r_collar','head','l_shoulder','r_shoulder',
    'l_elbow','r_elbow','l_wrist','r_wrist','l_hand','r_hand','jaw'
]
J = {name: idx for idx, name in enumerate(JOINT_NAMES)}

# (joint_A, vertex_B, joint_C)  →  angle at vertex B
ANGLE_TRIPLETS = {
    'left_knee_angle':     (J['l_hip'],      J['l_knee'],     J['l_ankle']),
    'right_knee_angle':    (J['r_hip'],      J['r_knee'],     J['r_ankle']),
    'left_hip_angle':      (J['spine1'],     J['l_hip'],      J['l_knee']),
    'right_hip_angle':     (J['spine1'],     J['r_hip'],      J['r_knee']),
    'left_ankle_angle':    (J['l_knee'],     J['l_ankle'],    J['l_foot']),
    'right_ankle_angle':   (J['r_knee'],     J['r_ankle'],    J['r_foot']),
    'left_elbow_angle':    (J['l_shoulder'], J['l_elbow'],    J['l_wrist']),
    'right_elbow_angle':   (J['r_shoulder'], J['r_elbow'],    J['r_wrist']),
    'left_shoulder_angle': (J['l_collar'],   J['l_shoulder'], J['l_elbow']),
    'right_shoulder_angle':(J['r_collar'],   J['r_shoulder'], J['r_elbow']),
    'trunk_flex':          (J['neck'],       J['spine2'],     J['pelvis']),
}

def compute_angle_3d(a, b, c):
    ba = a - b; bc = c - b
    cos_a = np.dot(ba, bc) / (np.linalg.norm(ba) * np.linalg.norm(bc) + 1e-8)
    return float(np.degrees(np.arccos(np.clip(cos_a, -1.0, 1.0))))

def compute_angles_sequence(joints_seq, triplets):
    result = {name: [] for name in triplets}
    for frame in joints_seq:
        for name, (ia, ib, ic) in triplets.items():
            result[name].append(compute_angle_3d(frame[ia], frame[ib], frame[ic]))
    return {k: np.array(v) for k, v in result.items()}

def load_joints3d(json_path):
    with open(json_path) as f:
        data = json.load(f)
    for key in [JOINTS_KEY, 'joints3d_25', 'joints3d', 'joints']:
        if key in data:
            arr = np.array(data[key])
            if arr.ndim == 3 and arr.shape[-1] == 3:
                return arr
    for val in data.values():
        if isinstance(val, list):
            arr = np.array(val)
            if arr.ndim == 3 and arr.shape[-1] == 3:
                return arr
    return None

print('✅ Joint definitions and utilities ready')

In [ ]:
# ── Compute angle distributions across all 8 train subjects ───────────────────
# Reads the locally downloaded JSON files. Takes ~5-15 min.

import time
from collections import defaultdict

angle_stats = defaultdict(lambda: defaultdict(list))  # {exercise: {angle_name: [values]}}
errors = []

t0 = time.time()
for subj in sorted(SUBJECTS.keys()):
    j3d_dir = Path(FIT3D_DIR) / subj / 'joints3d_25'
    files   = sorted(j3d_dir.glob('*.json'))
    print(f'{subj} ({len(files)} exercises) ...', end=' ', flush=True)
    for jf in files:
        exercise = jf.stem
        try:
            joints = load_joints3d(jf)
            if joints is None:
                errors.append(f'{subj}/{exercise}: parse failed'); continue
            angles = compute_angles_sequence(joints, ANGLE_TRIPLETS)
            for ang_name, vals in angles.items():
                angle_stats[exercise][ang_name].extend(vals.tolist())
        except Exception as e:
            errors.append(f'{subj}/{exercise}: {e}')
    print('done')

print(f'\n✅ Finished in {(time.time()-t0)/60:.1f} min')
print(f'   Exercises with data: {len(angle_stats)}')
if errors:
    print(f'   ⚠️  {len(errors)} errors:'); [print(f'      {e}') for e in errors[:5]]

In [ ]:
# ── Build ANGLE_RANGES and save ───────────────────────────────────────────────

ANGLE_RANGES = {}
for exercise, angle_dict in sorted(angle_stats.items()):
    ANGLE_RANGES[exercise] = {}
    for angle_name, values in sorted(angle_dict.items()):
        arr = np.array(values)
        ANGLE_RANGES[exercise][angle_name] = {
            'p5':  round(float(np.percentile(arr, 5)),  1),
            'p25': round(float(np.percentile(arr, 25)), 1),
            'p50': round(float(np.percentile(arr, 50)), 1),
            'p75': round(float(np.percentile(arr, 75)), 1),
            'p95': round(float(np.percentile(arr, 95)), 1),
            'min': round(float(arr.min()), 1),
            'max': round(float(arr.max()), 1),
            'n_frames': len(arr),
        }

os.makedirs('/kaggle/working/output', exist_ok=True)
out_path = '/kaggle/working/output/golden_angle_ranges.json'
with open(out_path, 'w') as f:
    json.dump(ANGLE_RANGES, f, indent=2)

print(f'✅ Saved → {out_path}')
print(f'   {len(ANGLE_RANGES)} exercises covered')

# Preview key exercises
for ex in ['squat', 'pushup', 'deadlift', 'dumbbell_biceps_curls']:
    if ex in ANGLE_RANGES:
        print(f'\n--- {ex} ---')
        for ang, stats in ANGLE_RANGES[ex].items():
            if any(k in ang for k in ['knee', 'elbow', 'hip']):
                print(f'  {ang:30s}  p5={stats["p5"]:5.1f}°  p50={stats["p50"]:5.1f}°  p95={stats["p95"]:5.1f}°')

---
## ✅ Step 12 — Validate Rep Counter with Fit3D Ground Truth

In [ ]:
# ── Explore rep_ann.json format ───────────────────────────────────────────────
with open(f'{FIT3D_DIR}/s03/rep_ann.json') as f:
    rep_sample = json.load(f)

print('rep_ann.json structure (s03):')
print(f'  Keys: {list(rep_sample.keys())[:6]} ... ({len(rep_sample)} total)')
for ex, val in list(rep_sample.items())[:2]:
    print(f'\n  [{ex}]')
    if isinstance(val, dict):
        for k, v in val.items(): print(f'    {k}: {v}')
    else:
        print(f'    value: {val}')

In [ ]:
from scipy.signal import find_peaks

def count_reps_from_angles(angle_seq, prominence=10, distance=15):
    if len(angle_seq) < 10: return 0, []
    peaks, _ = find_peaks(angle_seq, prominence=prominence, distance=distance)
    return len(peaks), peaks.tolist()

def get_gt_rep_count(entry):
    if isinstance(entry, int): return entry
    if isinstance(entry, dict):
        for key in ['reps', 'rep_count', 'count', 'n_reps', 'num_reps']:
            if key in entry: return int(entry[key])
        for key in ['intervals', 'segments', 'boundaries']:
            if key in entry: return len(entry[key])
    return None

def best_angle_for_exercise(exercise, angle_ranges):
    if exercise not in angle_ranges: return 'left_knee_angle'
    return max(angle_ranges[exercise].items(),
               key=lambda kv: kv[1]['p95'] - kv[1]['p5'])[0]

# ── Run validation ────────────────────────────────────────────────────────────
results = []
for subj in sorted(SUBJECTS.keys()):
    rep_file = Path(FIT3D_DIR) / subj / 'rep_ann.json'
    j3d_dir  = Path(FIT3D_DIR) / subj / 'joints3d_25'
    if not rep_file.exists(): continue

    rep_ann = json.load(open(rep_file))
    for jf in sorted(j3d_dir.glob('*.json')):
        exercise = jf.stem
        gt_entry = rep_ann.get(exercise)
        if gt_entry is None: continue
        gt_reps = get_gt_rep_count(gt_entry)
        if not gt_reps: continue

        joints = load_joints3d(jf)
        if joints is None: continue
        angles     = compute_angles_sequence(joints, ANGLE_TRIPLETS)
        best_ang   = best_angle_for_exercise(exercise, ANGLE_RANGES)
        angle_seq  = angles.get(best_ang, next(iter(angles.values())))
        pred_reps, _ = count_reps_from_angles(angle_seq)
        acc = max(0.0, 1.0 - abs(pred_reps - gt_reps) / gt_reps)
        results.append({'subject': subj, 'exercise': exercise,
                        'gt_reps': gt_reps, 'pred_reps': pred_reps,
                        'accuracy': round(acc, 3), 'angle_used': best_ang})

# ── Summary ───────────────────────────────────────────────────────────────────
accs = [r['accuracy'] for r in results]
mean_acc  = float(np.mean(accs))
pct_100   = sum(1 for a in accs if a == 1.0) / len(accs) * 100
pct_90    = sum(1 for a in accs if a >= 0.9)  / len(accs) * 100

print(f'=== Rep Counter Validation ({'n='+str(len(results))} sequences) ===')
print(f'  Mean accuracy:   {mean_acc:.3f}  ({mean_acc*100:.1f}%)')
print(f'  Perfect (100%):  {pct_100:.1f}% of sequences')
print(f'  ≥ 90% accuracy:  {pct_90:.1f}% of sequences')

from collections import defaultdict
per_ex = defaultdict(list)
for r in results: per_ex[r['exercise']].append(r['accuracy'])
print('\nPer-exercise mean (worst first):')
for ex, ex_accs in sorted(per_ex.items(), key=lambda kv: np.mean(kv[1]))[:10]:
    print(f'  {ex:40s}  {np.mean(ex_accs):.3f}  (n={len(ex_accs)})')

In [ ]:
# ── Save validation results ───────────────────────────────────────────────────
out = {
    'summary': {
        'n_sequences':   len(results),
        'mean_accuracy': round(mean_acc, 4),
        'pct_perfect':   round(pct_100, 1),
        'pct_gte90':     round(pct_90,  1),
    },
    'results': results,
}
val_path = '/kaggle/working/output/rep_counter_validation.json'
with open(val_path, 'w') as f:
    json.dump(out, f, indent=2)

print(f'✅ Saved → {val_path}')
print()
print('=== Final output files ===')
for p in ['/kaggle/working/output/golden_angle_ranges.json',
          '/kaggle/working/output/rep_counter_validation.json']:
    sz = os.path.getsize(p) / 1024
    print(f'  {p}  ({sz:.0f} KB)')

print()
print('💡 Download these from the Kaggle Output panel (right side → Output tab)')
print('   Then copy them to your Google Drive under GYMVISION AI/data/eval/')

---
## P02 Complete

| Thesis Metric | Output |
|---|---|
| MAE ≤ 5° (angle ground truth) | `golden_angle_ranges.json` |
| Rep counter accuracy | `rep_counter_validation.json` |

Download both files from the **Output tab** on the right, then upload them to your Drive.

### Troubleshooting
| Issue | Fix |
|---|---|
| `KeyError: 'GDRIVE_CLIENT_ID'` | Secret isn't toggled ON for this notebook — check Secrets panel |
| Auth URL doesn't work | Make sure you enabled the Drive API in Google Cloud Console |
| Joint shape wrong | Check the `JOINTS_KEY` printed in the sample cell and update manually |
| Rep counts way off | Adjust `prominence` or `distance` in `count_reps_from_angles()` |